# 06. Binary Search: Advanced Problems

Binary search on **partitions**, **values**, **unknown sizes**, **floating point**, and **unimodal functions**.

**Learning order:** MEDIAN OF TWO SORTED ARRAYS → KTH SMALLEST IN A MATRIX → DUPLICATES IN ROTATED → UNKNOWN SIZE → SINGLE ELEMENT → K CLOSEST → AGGRESSIVE COWS → SPLIT ARRAY → FLOATS AND TERNARY → INTERVIEW THINKING

## 1. Median of Two Sorted Arrays in O(log(min(m, n)))

Do **not** merge. Split both arrays so that the **left parts together** hold `(m + n + 1) // 2` elements, and every left element is `<=` every right element. Binary search the cut `i` in the **shorter** array (`j = half - i` is then fixed in the other).

A cut is correct when `A[i-1] <= B[j]` and `B[j-1] <= A[i]`. Then the median uses `max(left)` (and `min(right)` if the total is even). Use `-inf` / `+inf` for missing neighbours.

In [1]:
def find_median_sorted_arrays(a, b):
    if len(a) > len(b):
        a, b = b, a                                           # binary search the shorter array
    m, n = len(a), len(b)
    lo, hi, half = 0, m, (m + n + 1) // 2
    while lo <= hi:
        i = (lo + hi) // 2
        j = half - i
        a_left = a[i - 1] if i > 0 else float("-inf")
        a_right = a[i] if i < m else float("inf")
        b_left = b[j - 1] if j > 0 else float("-inf")
        b_right = b[j] if j < n else float("inf")
        if a_left <= b_right and b_left <= a_right:           # correct partition
            if (m + n) % 2:
                return float(max(a_left, b_left))
            return (max(a_left, b_left) + min(a_right, b_right)) / 2
        if a_left > b_right:
            hi = i - 1
        else:
            lo = i + 1

print(find_median_sorted_arrays([1, 3], [2]), find_median_sorted_arrays([1, 2], [3, 4]), find_median_sorted_arrays([], [1]))   # 2.0 2.5 1.0

2.0 2.5 1.0


## 2. Kth Smallest in a Sorted Matrix: binary search on the **value**

The answer lies between `matrix[0][0]` and `matrix[-1][-1]`. For a guess `mid`, **count** the elements `<= mid` with a staircase walk (O(n)). The smallest `mid` whose count is `>= k` is the answer. O(n log(max - min)).

In [2]:
def kth_smallest(matrix, k):
    n = len(matrix)
    def count_le(x):                                          # number of elements <= x
        count, r, c = 0, n - 1, 0
        while r >= 0 and c < n:
            if matrix[r][c] <= x:
                count += r + 1
                c += 1
            else:
                r -= 1
        return count
    lo, hi = matrix[0][0], matrix[-1][-1]
    while lo < hi:
        mid = (lo + hi) // 2
        if count_le(mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

print(kth_smallest([[1, 5, 9], [10, 11, 13], [12, 13, 15]], 8))   # 13

13


## 3. Search in Rotated Sorted Array II (duplicates)

With duplicates, `nums[lo] == nums[mid] == nums[hi]` hides which half is sorted. Safe move: **shrink both ends by one**. Worst case becomes **O(n)** (for example all equal), average O(log n).

In [3]:
def search_rotated_with_duplicates(nums, target):
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return True
        if nums[lo] == nums[mid] == nums[hi]:
            lo += 1
            hi -= 1
        elif nums[lo] <= nums[mid]:                           # left half sorted
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        else:                                                 # right half sorted
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return False

print(search_rotated_with_duplicates([2, 5, 6, 0, 0, 1, 2], 0), search_rotated_with_duplicates([2, 5, 6, 0, 0, 1, 2], 3))   # True False

True False


## 4. Search in a sorted array of **unknown size**

You can only call `get(i)` (returns a huge sentinel past the end). First find an **upper bound by doubling** (`1, 2, 4, 8, ...`) in O(log n), then binary search inside. Same idea as exponential search.

In [4]:
class ArrayReader:
    def __init__(self, data):
        self.data = data
    def get(self, i):
        return self.data[i] if i < len(self.data) else 2**31 - 1

def search_unknown_size(reader, target):
    hi = 1
    while reader.get(hi) < target:
        hi *= 2                                               # grow the window exponentially
    lo = hi // 2
    while lo <= hi:
        mid = (lo + hi) // 2
        v = reader.get(mid)
        if v == target:
            return mid
        if v < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

print(search_unknown_size(ArrayReader([-1, 0, 3, 5, 9, 12]), 9), search_unknown_size(ArrayReader([-1, 0, 3, 5, 9, 12]), 2))   # 4 -1

4 -1


## 5. Single Element in a Sorted Array (pairs)

Every element appears twice except one. Before the single element, pairs start at **even** indexes (`nums[2k] == nums[2k+1]`). After it, pairs start at odd indexes. Binary search on the first even index where the pair breaks.

In [5]:
def single_non_duplicate(nums):
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if mid % 2 == 1:
            mid -= 1                                          # make mid the start of a pair
        if nums[mid] == nums[mid + 1]:
            lo = mid + 2                                      # pair intact: single is on the right
        else:
            hi = mid
    return nums[lo]

print(single_non_duplicate([1, 1, 2, 3, 3, 4, 4, 8, 8]), single_non_duplicate([3, 3, 7, 7, 10, 11, 11]))   # 2 10

2 10


## 6. Find K Closest Elements (binary search the window start)

The answer is a window of size `k`. Binary search its **left end** `lo` in `[0, n - k]`. Compare the distance of `arr[mid]` (the window's left) and `arr[mid + k]` (the element just outside on the right) to `x`: if the left is farther, shift the window right.

In [6]:
def find_closest_elements(arr, k, x):
    lo, hi = 0, len(arr) - k
    while lo < hi:
        mid = (lo + hi) // 2
        if x - arr[mid] > arr[mid + k] - x:
            lo = mid + 1
        else:
            hi = mid
    return arr[lo:lo + k]

print(find_closest_elements([1, 2, 3, 4, 5], 4, 3), find_closest_elements([1, 2, 3, 4, 5], 4, -1))   # [1, 2, 3, 4] [1, 2, 3, 4]

[1, 2, 3, 4] [1, 2, 3, 4]


## 7. Aggressive Cows (maximize the minimum distance)

Place `c` cows in stalls (positions) so the **minimum distance between any two cows is as large as possible**. Binary search the distance `d` (last True template). `feasible(d)`: place the first cow at the leftmost stall, then greedily the next cow at the first stall at least `d` away.

In [7]:
def aggressive_cows(stalls, cows):
    stalls.sort()
    def feasible(d):
        placed, last = 1, stalls[0]
        for s in stalls[1:]:
            if s - last >= d:
                placed += 1
                last = s
        return placed >= cows
    lo, hi = 1, stalls[-1] - stalls[0]
    while lo < hi:
        mid = (lo + hi + 1) // 2                              # upper middle: maximize
        if feasible(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

print(aggressive_cows([1, 2, 4, 8, 9], 3))   # 3

3


## 8. Split Array Largest Sum (minimize the maximum)

Split into `m` contiguous parts to **minimize the largest part sum**. Binary search the answer between `max(nums)` and `sum(nums)`. `feasible(limit)`: greedily cut when adding the next number would exceed `limit`; the parts needed must be `<= m`.

In [8]:
def split_array(nums, m):
    def parts_needed(limit):
        parts, cur = 1, 0
        for x in nums:
            if cur + x > limit:
                parts += 1
                cur = 0
            cur += x
        return parts
    lo, hi = max(nums), sum(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if parts_needed(mid) <= m:
            hi = mid
        else:
            lo = mid + 1
    return lo

print(split_array([7, 2, 5, 10, 8], 2), split_array([1, 2, 3, 4, 5], 2))   # 18 9

18 9


## 9. Floating point binary search and ternary search

- **Real-valued answers:** loop a **fixed number of iterations** (e.g. 100) or until `hi - lo < eps`. Never compare floats with `==`.
- **Ternary search** finds the maximum (or minimum) of a **unimodal** function (increases then decreases). Compare two interior points and discard one third.

In [9]:
def sqrt_float(x, iterations=100):
    lo, hi = 0.0, max(1.0, x)
    for _ in range(iterations):
        mid = (lo + hi) / 2
        if mid * mid <= x:
            lo = mid
        else:
            hi = mid
    return lo

def ternary_max(f, lo, hi, iterations=200):
    for _ in range(iterations):
        m1, m2 = lo + (hi - lo) / 3, hi - (hi - lo) / 3
        if f(m1) < f(m2):
            lo = m1                                           # the peak is to the right of m1
        else:
            hi = m2
    return (lo + hi) / 2

print(abs(sqrt_float(2) - 2 ** 0.5) < 1e-9, abs(ternary_max(lambda x: -(x - 3) ** 2 + 5, 0, 10) - 3) < 1e-6)   # True True

True True


## 10. Interview thinking

**Signals:** "two sorted arrays median in log time", "k-th smallest in sorted rows/columns", "unknown size", "minimize the maximum / maximize the minimum", "real number answer", "unimodal".

**Say:** "I binary search the partition / the answer. For each guess I verify feasibility in O(n), so total O(n log range)."

**Common mistakes:** binary searching the **longer** array in the median problem, missing `±inf` sentinels, lower middle in a maximize search (infinite loop), float `==` comparisons, wrong range for the answer (`max(nums)` to `sum(nums)`).

## Quick Revision

- Median of two arrays: search the cut in the shorter array, check `A[i-1] <= B[j]` and `B[j-1] <= A[i]`.
- Kth in matrix: search the **value**, count `<= mid` with a staircase.
- Duplicates break rotated search → shrink both ends, worst O(n).
- Unknown size: double, then binary search.
- Minimize-max / maximize-min: binary search the answer + greedy feasibility.
- Floats: fixed iterations. Ternary for unimodal.